In [1]:
import re
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

dataset = "hf://datasets/PaulineSanchez/Translation_words_and_sentences_english_french/data/train-00000-of-00001-3d14582ea46e1b17.parquet"
df = pd.read_parquet(dataset)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

!pip install sentencepiece
import sentencepiece as spm

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


device: cuda


In [2]:
c1, c2 = df.columns
enc = np.array(df[c1])
dec = np.array(df[c2])

for i in range(len(enc)):
  enc[i] = enc[i].lower()
  dec[i] = dec[i].lower()

for i in np.random.choice(np.arange(len(df)), 10):
  print(f"{enc[i]:<50} | {dec[i]}")

i don't normally do this.                          | normalement, je ne fais pas ça.
i suppose you enjoy that sort of thing.            | je suppose que vous prenez plaisir à cette sorte de chose.
do you want to know who helped?                    | veux-tu savoir qui a aidé ?
i need some sugar to make a cake.                  | j'ai besoin de sucre pour faire un gâteau.
the postman was bitten by that dog.                | le facteur a été mordu par ce chien.
how long has it been since you played with a yo-yo? | depuis quand n'as-tu pas joué au yoyo ?
i declined his invitation to dinner.               | je refusai son invitation à dîner.
i don't think what you're doing is legal.          | je ne pense pas que ce que tu es en train de faire soit légal.
i'm tired of doing this.                           | je suis fatiguée de faire ceci.
have you been up all night?                        | as-tu été debout toute la nuit ?


In [3]:
with open("bpe_train.txt", "w", encoding="utf-8") as f:
    for s in enc:
        f.write(str(s).lower() + "\n")
    for s in dec:
        f.write(str(s).lower() + "\n")

spm.SentencePieceTrainer.train(
    input="bpe_train.txt",
    model_prefix="bpe",
    vocab_size=8000,
    model_type="bpe",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

sp = spm.SentencePieceProcessor()
sp.load("bpe.model")

True

In [4]:
""" Hyperparameters """

embedding_dim = 300
hidden_size = 300 # context vector length
max_decoder_seq = 50
batch_size = 128

lr = 0.001
epochs = 10

pad_tok = sp.pad_id()
unk_tok = sp.unk_id()
sos_tok = sp.bos_id()
eos_tok = sp.eos_id()

vocab_len = sp.get_piece_size()
enc_vocab_len = vocab_len
dec_vocab_len = vocab_len

In [5]:
E = []
for sentence in enc:
    ids = sp.encode(str(sentence).lower(), out_type=int)
    ids = [sos_tok] + ids + [eos_tok]
    E.append(torch.tensor(ids, dtype=torch.long))

D = []
for sentence in dec:
    ids = sp.encode(str(sentence).lower(), out_type=int)
    ids = [sos_tok] + ids + [eos_tok]
    D.append(torch.tensor(ids, dtype=torch.long))

enc_pad_tok = pad_tok
dec_pad_tok = pad_tok

In [6]:
class GRUSeq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_embed = nn.Embedding(vocab_len, embedding_dim, padding_idx=pad_tok)
        self.dec_embed = nn.Embedding(vocab_len, embedding_dim, padding_idx=pad_tok)

        """ encoder/decoder GRU """
        self.encoder = nn.GRU(embedding_dim, hidden_size, batch_first=True)
        self.decoder = nn.GRU(embedding_dim, hidden_size, batch_first=True)

        self.fc = nn.Linear(hidden_size, vocab_len)

    def forward(self, e, d=None):
        s = torch.zeros(1, e.size(0), hidden_size, device=e.device)
        w = self.enc_embed(e)
        _, s = self.encoder(w, s)

        if d is None:
            tok = torch.full((e.size(0), 1), sos_tok, dtype=torch.long, device=e.device)
            outputs = []
            for _ in range(max_decoder_seq):
                w = self.dec_embed(tok)
                out, s = self.decoder(w, s)
                logits = self.fc(out)

                outputs.append(logits)
                tok = torch.argmax(logits[:, -1, :], dim=1).unsqueeze(1)

                if torch.all(tok.squeeze(1) == eos_tok):
                    break

            return torch.cat(outputs, dim=1)

        w = self.dec_embed(d)
        out, _ = self.decoder(w, s)
        out = self.fc(out)

        return out

gru_seq2seq = GRUSeq2Seq().to(device)

In [7]:
class Seq2SeqDataset(Dataset):
  def __init__(self, e, d):
    self.e = e
    self.d = d

  def __getitem__(self, index):
     return self.e[index], self.d[index]

  def __len__(self):
    return len(self.e)

""" collate function for batching """
def collate_fn(batch):
  e, d = zip(*batch)

  max_e = max(len(seq) for seq in e)
  max_d = max(len(seq) for seq in d)

  pad_e = torch.zeros(len(e), max_e, dtype=torch.long)
  pad_d = torch.zeros(len(d), max_d, dtype=torch.long)

  for i in range(len(e)):
    pad_e[i] = torch.cat([e[i], torch.tensor([enc_pad_tok] * (max_e - len(e[i])))])

  for i in range(len(d)):
    pad_d[i] = torch.cat([d[i], torch.tensor([dec_pad_tok] * (max_d - len(d[i])))])

  return pad_e, pad_d

In [8]:
dataset = Seq2SeqDataset(E, D)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

criterion = nn.CrossEntropyLoss(ignore_index=dec_pad_tok)
optimizer = optim.Adam(gru_seq2seq.parameters(), lr=lr)

for epoch in range(epochs):
  total_loss = 0.0
  for e, d in dataloader:
    e = e.to(device)
    d = d.to(device)

    optimizer.zero_grad()

    d_in, d_out = d[:, :-1], d[:, 1:]
    out = gru_seq2seq(e[:, :-1], d_in)

    loss = criterion(
        out.reshape(-1, out.size(-1)),
        d_out.reshape(-1)
    )

    total_loss += loss.item()

    loss.backward()
    optimizer.step()

  print(f"Epoch: {epoch+1}/{epochs} | loss: {total_loss / len(dataloader)}")

KeyboardInterrupt: 

In [ ]:
text = """
Yesterday I went to the market with my sister. We bought fresh bread, fruit, and some cheese for dinner.
The weather was cold, but the streets were full of people. After lunch, we visited a small bookstore near the river and
spent almost an hour looking at old novels. I was tired when I got home, but I still watched a movie before going to sleep.
"""

text = sent_tokenize(text)

for test in text:
  seq = sp.encode(test.lower(), out_type=int)
  seq = [sos_tok] + seq + [eos_tok]

  gru_seq2seq.eval()

  with torch.no_grad():
      seq = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)

      pred = gru_seq2seq(seq)

      pred_ids = []

      for prob in pred[0]:
          pred_id = torch.argmax(prob).item()

          if pred_id == eos_tok:
              break

          if pred_id not in [sos_tok, pad_tok]:
              pred_ids.append(pred_id)

      print(sp.decode(pred_ids))
  print()